# K-Fold Dataset Creation (Clean & User Order Preserved) ✅

## 🎯 Goals
1. Generate **clean CSV files** without phantom 'item_id' column
2. **Preserve user ordering** (numerical order: 1, 2, 3, ... not lexicographic: 1, 10, 100, ...)
3. User-wise K-fold split with repair (all users/items have ≥1 train observation)
4. **Update existing movielenz_data folder** (similarity files remain unchanged)

## 🔧 Fixes Applied
- Clear `index.name` and `columns.name` before saving CSV
- Convert user IDs to integers for numerical sorting
- Maintain consistent user ordering across all folds

## 📂 Output Structure
```
movielenz_data/
  original_matrix.csv     # Clean (1682, 943) no 'item_id'
  fold_01/
    train.csv            # Updated with clean data
    test.csv             # Updated with clean data
    similarity.npy       # Unchanged (verified correct)
  fold_02/
    ...
```

## ✅ Similarity Files
Precomputed similarity files are **verified correct** and will remain unchanged.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

## Step 1: Load Original MovieLens Data

In [ ]:
# Load raw MovieLens data
url = "https://raw.githubusercontent.com/park61/imputation/main/u.data"
colnames = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv(url, sep='\t', names=colnames, header=None)
print(f"Raw data shape: {df.shape}")
print(f"Users: {df['user_id'].nunique()}, Items: {df['item_id'].nunique()}")
print(f"Ratings: {len(df)}")

## Step 2: Pivot to Item × User Matrix with Numerical User Order

In [ ]:
# Pivot: item × user (items as rows, users as columns)
pivot_df = df.pivot_table(index="item_id", columns="user_id", values="rating", aggfunc='mean')

print(f"\nPivot table shape: {pivot_df.shape}")
print(f"Index (items): {len(pivot_df.index)}")
print(f"Columns (users): {len(pivot_df.columns)}")
print(f"Index name: '{pivot_df.index.name}'")
print(f"Columns name: '{pivot_df.columns.name}'")

# Check user ordering
print(f"\nFirst 20 user IDs: {pivot_df.columns[:20].tolist()}")
print(f"User ID types: {type(pivot_df.columns[0])}")

# Users should already be in numerical order (1, 2, 3, ...) from pivot_table
# But let's ensure by sorting columns numerically
pivot_df = pivot_df.sort_index(axis=1)  # Sort columns (users) numerically
pivot_df = pivot_df.sort_index(axis=0)  # Sort rows (items) numerically

print(f"\nAfter sorting - First 20 user IDs: {pivot_df.columns[:20].tolist()}")

## Step 3: Save Clean Original Matrix

In [ ]:
# 🔧 Bug fix: Clear index.name and columns.name before saving
pivot_df_clean = pivot_df.copy()
pivot_df_clean.index.name = None
pivot_df_clean.columns.name = None

# Save to existing directory (will update)
output_dir = Path("../data/movielenz_data")
output_dir.mkdir(parents=True, exist_ok=True)

original_matrix_path = output_dir / "original_matrix.csv"
pivot_df_clean.to_csv(original_matrix_path)

print(f"✅ Saved clean original matrix: {original_matrix_path}")
print(f"   Shape: {pivot_df_clean.shape}")
print(f"   Index name: {pivot_df_clean.index.name}")
print(f"   Columns name: {pivot_df_clean.columns.name}")

# Verify by reloading
verify_df = pd.read_csv(original_matrix_path, index_col=0)
print(f"\n🔍 Verification after reload:")
print(f"   Shape: {verify_df.shape}")
print(f"   First column name: '{verify_df.columns[0]}'")
print(f"   First 10 columns: {verify_df.columns[:10].tolist()}")
if 'item_id' in verify_df.columns:
    print(f"   ❌ ERROR: 'item_id' found in columns!")
else:
    print(f"   ✅ Clean: No 'item_id' column")

## Step 4: K-Fold Split Function (User-wise with Repair)

In [ ]:
def create_kfold_datasets_userwise(df: pd.DataFrame, k: int, seed: int = 42, repair: bool = True):
    """
    Create user-wise K-fold splits on an item×user rating matrix.
    - Exclusive test partitions: each observed (item,user) appears in test of exactly one fold
    - Optional repair: ensure every user & item has ≥1 train observation in each fold
    Returns: list of dicts [{ 'train': DataFrame, 'test': DataFrame, 'test_size': int, 'sparsity_train': float }, ...]
    """
    rng = np.random.default_rng(seed)
    n_row, n_col = df.shape

    # 1) collect observed row indices for each user (column)
    by_user = {}
    for u in range(n_col):
        rows_u = np.where(df.iloc[:, u].notna().to_numpy())[0]
        arr = rows_u.copy()
        rng.shuffle(arr)  # deterministic with seed
        # Evenly split this user's observed rows into k parts
        by_user[u] = [np.array(x, dtype=int) for x in np.array_split(arr, k)]

    # 2) build test index sets per fold (as pairs of (row,item))
    test_pairs = [set() for _ in range(k)]
    for u in range(n_col):
        for f in range(k):
            for r in by_user[u][f]:
                test_pairs[f].add((r, u))

    # Precompute totals for users/items
    user_total = np.zeros(n_col, dtype=int)
    item_total = np.zeros(n_row, dtype=int)
    obs = np.where(df.notna().to_numpy())
    for r, c in zip(*obs):
        user_total[c] += 1
        item_total[r] += 1

    def repair_fold(tp: set):
        """Move minimal number of (r,c) from test -> train so that every user & item has ≥1 train obs."""
        # counts in test
        user_test = np.zeros(n_col, dtype=int)
        item_test = np.zeros(n_row, dtype=int)
        for r, c in tp:
            user_test[c] += 1
            item_test[r] += 1
        user_train = user_total - user_test
        item_train = item_total - item_test

        # (a) users with 0 train observations -> move one of their test entries back
        users_zero = np.where(user_train == 0)[0]
        for u in users_zero:
            cand = [(r, c) for (r, c) in tp if c == u]
            if cand:
                r, c = cand[0]
                tp.remove((r, c))
                user_test[c] -= 1
                item_test[r] -= 1
                user_train[c] += 1
                item_train[r] += 1

        # (b) items with 0 train observations -> move one of their test entries back
        items_zero = np.where(item_train == 0)[0]
        for r0 in items_zero:
            cand = [(r, c) for (r, c) in tp if r == r0]
            if cand:
                r, c = cand[0]
                tp.remove((r, c))
                user_test[c] -= 1
                item_test[r] -= 1
                user_train[c] += 1
                item_train[r] += 1

        return tp

    folds = []
    for f in range(k):
        tp = test_pairs[f].copy()
        if repair:
            tp = repair_fold(tp)

        # 3) materialize train / test matrices
        train = df.copy()
        test = pd.DataFrame(np.nan, index=df.index, columns=df.columns)

        if tp:
            rows, cols = zip(*tp)
            rows = np.array(rows, dtype=int)
            cols = np.array(cols, dtype=int)
            # fill test with original values, and mask train at those positions
            test.values[(rows, cols)] = df.values[(rows, cols)]
            train.values[(rows, cols)] = np.nan

        sparsity_train = 1.0 - np.nanmean(train.notna().to_numpy())
        folds.append({
            'train': train,
            'test': test,
            'test_size': len(tp),
            'sparsity_train': sparsity_train
        })
        print(f"Fold {f+1}: test_size={len(tp)}, train_sparsity={sparsity_train:.4f}")

    return folds

## Step 5: Save Function with Clean CSV

In [ ]:
def save_kfold_datasets_clean(folds, save_path: str = "../data/movielenz_data"):
    """
    Save each fold's train/test as CLEAN CSV files:
      - No 'item_id' phantom column
      - Numerical user ordering preserved
      - Clear index.name and columns.name
      - Updates existing folder (similarity.npy files remain unchanged)
    """
    base = Path(save_path)
    base.mkdir(parents=True, exist_ok=True)

    k = len(folds)
    for i, FT in enumerate(folds, start=1):
        fold_dir = base / f"fold_{str(i).zfill(2)}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        
        # 🔧 Bug fix 1: Clear index.name and columns.name before saving
        train_df = FT['train'].copy()
        test_df = FT['test'].copy()
        
        train_df.index.name = None
        train_df.columns.name = None
        test_df.index.name = None
        test_df.columns.name = None
        
        # Save as CSV (index=True to preserve item IDs as first column)
        # This will overwrite existing train.csv and test.csv
        train_path = fold_dir / "train.csv"
        test_path = fold_dir / "test.csv"
        
        train_df.to_csv(train_path)
        test_df.to_csv(test_path)
        
        print(f"Updated: {train_path} | {test_path}")
        
        # Check if similarity.npy exists
        sim_path = fold_dir / "similarity.npy"
        if sim_path.exists():
            print(f"  ✓ Similarity file preserved: {sim_path}")
        
        # Verify clean save
        if i == 1:  # Check first fold
            verify_train = pd.read_csv(train_path, index_col=0)
            print(f"  ✓ Verified fold 1 train: shape={verify_train.shape}")
            print(f"    First 10 columns: {verify_train.columns[:10].tolist()}")
            if 'item_id' in verify_train.columns:
                print(f"    ❌ ERROR: 'item_id' in columns!")
            else:
                print(f"    ✅ Clean: No 'item_id' column")

## Step 6: Generate K-Fold Datasets

In [ ]:
# Generate 10-fold datasets
k = 10
seed = 2025

print(f"\n{'='*80}")
print(f"Creating {k}-fold datasets (user-wise split, seed={seed})")
print(f"{'='*80}\n")

folds = create_kfold_datasets_userwise(pivot_df_clean, k=k, seed=seed, repair=True)

## Step 7: Save All Folds

In [ ]:
print(f"\n{'='*80}")
print(f"Updating folds in: movielenz_data/")
print(f"  (train.csv and test.csv will be updated)")
print(f"  (similarity.npy files will remain unchanged)")
print(f"{'='*80}\n")

save_kfold_datasets_clean(folds, save_path="./movielenz_data")

print(f"\n{'='*80}")
print(f"✅ All folds updated successfully!")
print(f"✅ Similarity files preserved (verified correct)")
print(f"{'='*80}")

## Step 8: Final Verification

In [ ]:
# Comprehensive verification
print(f"\n{'='*80}")
print(f"🔍 FINAL VERIFICATION")
print(f"{'='*80}\n")

# Check fold_01/train.csv
train_csv = "../data/movielenz_data/fold_01/train.csv"

# Method 1: Without index_col (should show 944 if buggy, 943+1 header if clean)
df_raw = pd.read_csv(train_csv)
print(f"1. Raw CSV (no index_col):")
print(f"   Shape: {df_raw.shape}")
print(f"   First column: '{df_raw.columns[0]}'")
print(f"   Has 'item_id' column: {'item_id' in df_raw.columns}")

# Method 2: With index_col=0 (standard loading)
df_indexed = pd.read_csv(train_csv, index_col=0)
print(f"\n2. Standard load (index_col=0):")
print(f"   Shape: {df_indexed.shape}")
print(f"   Index name: {df_indexed.index.name}")
print(f"   Columns name: {df_indexed.columns.name}")
print(f"   First 20 columns: {df_indexed.columns[:20].tolist()}")

# Method 3: Check user ordering
user_ids = df_indexed.columns.tolist()
user_ids_int = [int(x) for x in user_ids]
is_sorted_numerically = user_ids_int == sorted(user_ids_int)

print(f"\n3. User ordering check:")
print(f"   Total users: {len(user_ids)}")
print(f"   User IDs (first 20): {user_ids[:20]}")
print(f"   Numerically sorted: {is_sorted_numerically} ✅" if is_sorted_numerically else f"   Numerically sorted: {is_sorted_numerically} ❌")

# Method 4: Check similarity file exists
import os
sim_file = "../data/movielenz_data/fold_01/similarity.npy"
sim_exists = os.path.exists(sim_file)
print(f"\n4. Similarity file check:")
print(f"   Path: {sim_file}")
print(f"   Exists: {sim_exists} ✅" if sim_exists else f"   Exists: {sim_exists} ❌")
if sim_exists:
    sim_array = np.load(sim_file)
    print(f"   Shape: {sim_array.shape}")
    print(f"   Expected: (943, 943)")

# Final verdict
print(f"\n{'='*80}")
if df_indexed.shape == (1682, 943) and 'item_id' not in df_raw.columns and is_sorted_numerically:
    print(f"✅✅✅ PERFECT! All checks passed:")
    print(f"  ✅ Shape: (1682, 943)")
    print(f"  ✅ No 'item_id' phantom column")
    print(f"  ✅ Numerical user ordering (1, 2, 3, ...)")
    if sim_exists:
        print(f"  ✅ Similarity file preserved")
else:
    print(f"❌ Issues detected:")
    if df_indexed.shape != (1682, 943):
        print(f"  ❌ Wrong shape: {df_indexed.shape}")
    if 'item_id' in df_raw.columns:
        print(f"  ❌ 'item_id' column present")
    if not is_sorted_numerically:
        print(f"  ❌ User ordering not numerical")
print(f"{'='*80}")